In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import joblib
import warnings
warnings.filterwarnings("ignore")

# ---------------------------
# 1) Gabor features
# ---------------------------
def extract_gabor_features(gray):
    features = []
    num_orientations = 4
    for theta in range(num_orientations):
        theta_val = theta / num_orientations * np.pi
        kernel = cv2.getGaborKernel((7,7), 4.0, theta_val, 10.0, 0.5, 0, ktype=cv2.CV_32F)
        fimg = cv2.filter2D(gray, cv2.CV_8UC3, kernel)
        features.extend([np.mean(fimg), np.std(fimg)])
    return np.array(features)  # length 8

# ---------------------------
# 2) Load dataset + color + gabor
# ---------------------------
def load_image_dataset(base_path, img_size=(128, 128)):
    X, y = [], []
    class_names = sorted([d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))])

    for label in tqdm(class_names, desc="Loading images"):
        class_dir = os.path.join(base_path, label)
        for img_file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_file)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue
                img = cv2.resize(img, img_size)

                # Convert color spaces (note: OpenCV default BGR)
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
                img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
                img_hsl = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HLS)
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

                # color means
                rgb_mean = np.mean(img_rgb.reshape(-1,3), axis=0)
                hsv_mean = np.mean(img_hsv.reshape(-1,3), axis=0)
                lab_mean = np.mean(img_lab.reshape(-1,3), axis=0)
                hsl_mean = np.mean(img_hsl.reshape(-1,3), axis=0)

                gabor_feats = extract_gabor_features(gray)

                feature_vector = np.concatenate([
                    rgb_mean,    # 3
                    hsv_mean,    # 3
                    hsl_mean,    # 3
                    lab_mean,    # 3
                    gabor_feats  # 8
                ])

                X.append(feature_vector)
                y.append(label)
            except Exception as e:
                print("⚠ Error membaca:", img_path, e)
                continue

    return np.array(X), np.array(y)

# ---------------------------
# Main training flow
# ---------------------------
if __name__ == "__main__":
    base_dataset = "./dataset_kain"
    img_size = (128, 128)

    X, y = load_image_dataset(base_dataset, img_size=img_size)
    print("Shape fitur:", X.shape)
    print("Jumlah data:", len(y))

    if len(y) == 0:
        raise RuntimeError("Dataset kosong atau path salah.")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    print(f"Jumlah data: {len(X)}, Fitur: {X.shape[1]}, Kelas: {len(np.unique(y))}")

    # pipeline factory
    def make_pipeline(mode="pca+lda", X_train=None, y_train=None, n_pca=10, n_lda=5):
        steps = [('scaler', StandardScaler())]

        n_pca = min(n_pca, X_train.shape[1])
        n_classes = len(np.unique(y_train))
        n_lda = max(1, min(n_lda, n_classes - 1))

        if mode == "pca":
            steps.append(('pca', PCA(n_components=n_pca, random_state=42)))
        elif mode == "lda":
            steps.append(('lda', LDA(n_components=n_lda)))
        elif mode == "pca+lda":
            steps.append(('pca', PCA(n_components=n_pca, random_state=42)))
            steps.append(('lda', LDA(n_components=n_lda)))

        steps.append(('svm', SVC(probability=True, class_weight='balanced')))
        return Pipeline(steps)

    def top2_accuracy(model, X, y_true):
        prob = model.predict_proba(X)
        top2 = np.argsort(prob, axis=1)[:, -2:]
        classes = model.classes_
        top2_labels = classes[top2]
        correct = sum([y_true[i] in top2_labels[i] for i in range(len(y_true))])
        return correct / len(y_true)

    param_grid = {
        'svm__C': [10, 50],
        'svm__kernel': ['rbf'],
        'svm__gamma': [1, 10]
    }

    mode = "pca+lda"
    pipeline = make_pipeline(mode, X_train, y_train, n_pca=10, n_lda=5)
    grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)
    grid.fit(X_train, y_train)

    # evaluation
    y_pred = grid.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    top2_acc = top2_accuracy(grid.best_estimator_, X_test, y_test)

    print(f"✅ Akurasi Top-1 : {acc:.4f}")
    print(f"🎯 Akurasi Top-2 : {top2_acc:.4f}")
    print("Best Params:", grid.best_params_)
    print(classification_report(y_test, y_pred))

    # Save final pipeline (includes scaler + pca/lda + svm)
    os.makedirs("./svm_results", exist_ok=True)
    model_path = "./svm_results/pipeline_model.pkl"
    joblib.dump(grid.best_estimator_, model_path)
    print("Model saved to:", model_path)

    # summary csv
    df = pd.DataFrame([{"mode": mode, "acc": acc, "top2_acc": top2_acc, "path": model_path}])
    df.to_csv("./svm_results/results_summary.csv", index=False)
    print("✅ Semua model disimpan.")


Loading images: 100%|████████████████████████████████████████████████████████████████| 200/200 [10:16<00:00,  3.08s/it]


Shape fitur: (9971, 20)
Jumlah data: 9971
Jumlah data: 9971, Fitur: 20, Kelas: 200
Fitting 5 folds for each of 4 candidates, totalling 20 fits
✅ Akurasi Top-1 : 0.7990
🎯 Akurasi Top-2 : 0.9023
Best Params: {'svm__C': 50, 'svm__gamma': 1, 'svm__kernel': 'rbf'}
              precision    recall  f1-score   support

         001       0.70      0.70      0.70        10
         003       0.89      0.80      0.84        10
         005       0.77      1.00      0.87        10
         007       0.86      0.60      0.71        10
         008       0.86      0.60      0.71        10
         009       0.18      0.40      0.25        10
         011       1.00      1.00      1.00        10
        011B       1.00      0.80      0.89        10
        037B       0.69      0.90      0.78        10
         046       0.90      0.90      0.90        10
         067       0.47      0.80      0.59        10
         072       1.00      0.60      0.75        10
         077       0.71      1.00    